## read cyc file

In [1]:
import re
def extract_id_from_experiment_name(experiment_name: str):
    # 첫 글자가 영어(a-zA-Z)이고, 그 뒤에 영어나 숫자가 9개 이상 연속되는 패턴을 찾습니다.
    # 즉, 전체 ID는 최소 10자 이상입니다.
    match = re.search(r"[a-zA-Z][a-zA-Z0-9]{9,}", experiment_name)
    if match:
        return match.group(0)
    else:
        return "None"
    
extract_id_from_experiment_name("07002345_240425_JF2_DV추가빌드_Ref_25oC_0.25CP_FDBD110401.cts")

'FDBD110401'

In [2]:
import struct
import datetime
import pandas as pd

In [19]:
def _read_ps_file_id_header(file):
    # Read and parse PS_FILE_ID_HEADER
    szFileID, szFileVersion = struct.unpack('II', file.read(8))
    szCreateDateTime = file.read(64).decode('utf-8',errors='replace').rstrip('\x00')
    szDescription = file.read(128).decode('utf-8',errors='replace').rstrip('\x00')
    szReserved = file.read(128).decode('utf-8',errors='replace').rstrip('\x00')

    return {
        'szFileID': szFileID,
        'szFileVersion': szFileVersion,
        'szCreateDateTime': szCreateDateTime,
        'szDescription': szDescription,
        'szReserved': szReserved,
    }

def _read_ps_record_file_header(file):
    # Read and parse PS_RECORD_FILE_HEADER'
    nColumCount = struct.unpack('i', file.read(4))[0]
    awColumnItem = struct.unpack(f'{nColumCount}H', file.read(nColumCount * 2))  # WORD is 2 bytes, assuming 42 columns

    # Mapping the column items to heir respective names
    column_mapping = {
            0x00: 'PS_STATE', 0x01: 'PS_VOLTAGE', 0x02: 'PS_CURRENT', 0x03: 'PS_CAPACITY',
            0x04: 'PS_IMPEDANCE', 0x05: 'PS_CODE', 0x06: 'PS_STEP_TIME', 0x07: 'PS_TOT_TIME',
            0x08: 'PS_GRADE_CODE', 0x09: 'PS_STEP_NO', 0x0A: 'PS_WATT', 0x0B: 'PS_WATT_HOUR',
            0x0C: 'PS_TEMPERATURE', 0x0D: 'PS_PRESSURE', 0x0E: 'PS_STEP_TYPE', 0x0F: 'PS_CUR_CYCLE',
            0x10: 'PS_TOT_CYCLE', 0x11: 'PS_TEST_NAME', 0x12: 'PS_SCHEDULE_NAME', 0x13: 'PS_CHANNEL_NO',
            0x14: 'PS_MODULE_NO', 0x15: 'PS_LOT_NO', 0x16: 'PS_DATA_SEQ', 0x17: 'PS_AVG_CURRENT',
            0x18: 'PS_AVG_VOLTAGE', 0x19: 'PS_CAPACITY_SUM', 0x1A: 'PS_CHARGE_CAP', 0x1B: 'PS_DISCHARGE_CAP',
            0x1C: 'PS_METER_DATA', 0x1D: 'PS_START_TIME', 0x1E: 'PS_END_TIME', 0x1F: 'PS_SHARING_INFO',
            0x20: 'PS_GOTO_COUNT', 0x21: 'PS_WATTHOUR_SUM', 0x22: 'PS_CHAR_WATTHOUR',
            0x23: 'PS_DISCHAR_WATTHOUR', 0x24: 'PS_INTEGRAL_CAPACITY', 0x25: 'PS_INTEGRAL_WATTHOUR',
            0x26: 'PS_CV_END_TIME', 0x27: 'PS_CYCLE_NUM', 0x28: 'PS_TOT_TIME_CARRY',
            0x29: 'PS_FARAD', 0x2A: 'PS_TEMPERATURE2', 0x2B: 'PS_DQDV', 0x2C: 'PS_CHARGE_CC_CAP',
            0x2D: 'PS_CHARGE_CV_CAP', 0x2E: 'PS_DISCHARGE_CC_CAP', 0x2F: 'PS_DISCHARGE_CV_CAP',
            0x30: 'PS_REALDATE', 0x31: 'PS_REALCLOCK', 0x32: 'PS_CHAMBER_TEMPERATURE',
            0x33: 'PS_AUX_TEMPERATURE', 0x34: 'PS_AUX_VOLTAGE', 0x3B: 'PS_AUX_THICKNESS2',
            0x3C: 'PS_AUX_PRESSURE1', 0x3D: 'PS_AUX_PRESSURE2', 0x3E: 'PS_AUX_PRESSURE3',
            0x3F: 'PS_AUX_PRESSURE4', 0x40: 'PS_AUX_THICKNESS1', 0x41: 'PS_GAS_CO2', 0x42: 'PS_GAS_TEMP', 
            0x43: 'PS_GAS_AH',  0x44: 'PS_GAS_BASELINE', 0x45: 'PS_GAS_TVOC', 0x46: 'PS_GAS_ETHANOL', 
            0x47: 'PS_GAS_H2', 0x49: 'PS_AMBIENT_TEMP', 0x4A: 'PS_GAS_VOLTAGE', 0x4B: 'PS_IMPEDANCE_100MS', 
            0x4C: 'PS_IMPEDANCE_1S', 0x4D: 'PS_IMPEDANCE_5S', 0x4E: 'PS_IMPEDANCE_30S', 0x4F: 'PS_IMPEDANCE_60S',
            0x70: 'PS_RESTORED_FLAG', 0x68: 'PS_CHAMBER_TEMP_SV', 0x69: 'PS_CHILER_TEMP_PV',
            0x6A: 'PS_CHILER_TEMP_SV', 0x6B: 'PS_CHILER_PUMP_PV', 0x6C: 'PS_CHILER_PUMP_SV',
            0x6D: 'PS_AUX_THICKNESS3', 0x6E: 'PS_AUX_THICKNESS4'
        }

    column_names = [column_mapping.get(item, f'UNKNOWN_0x{item:02X}') for item in awColumnItem[:nColumCount]]
    
    awColumnItem_dict = dict(zip(column_names, awColumnItem[:nColumCount]))

    return {
        'nColumCount': nColumCount,
        'awColumnItem': awColumnItem_dict,
        'columnNames': column_names
    }

def _read_record_data(file, nColumCount):
    # Read the record data
    data = []
    while True:
        record = file.read(nColumCount * 4)  # float is 4 bytes
        if not record:
            break
        if len(record) < nColumCount * 4: # 파일 끝에서 불완전한 레코드가 읽히는 경우 방지
            if len(record) == 0 and not data: # 파일이 비어있거나 헤더만 있는 경우
                break
            elif len(record) == 0: # 정상적으로 파일 끝에 도달한 경우
                break
            print(f"Warning: Incomplete record found at the end of the file. Expected {nColumCount * 4} bytes, got {len(record)}. Skipping.")
            break # 불완전한 레코드는 처리하지 않고 종료
        try:
            awColumnItem = struct.unpack(f'{nColumCount}f', record)
            data.append(awColumnItem)
        except struct.error as e:
            print(f"Error unpacking record: {e}. Record length: {len(record)}. Expected for {nColumCount} floats.")
            raise e 

    return data

def _preprocess_data(data: list, column_names: list)-> pd.DataFrame:
    filtered_data = [record for record in data]

    # Convert filtered data to DataFrame
    df = pd.DataFrame(filtered_data, columns=column_names)

    # Divide PS_VOLTAGE, PS_CAPACITY, and PS_CURRENT by 1000 if they exist in the DataFrame
    for column in ['PS_VOLTAGE', 'PS_CAPACITY', 'PS_CURRENT', 'PS_WATT', 'PS_WATT_HOUR', 'PS_CHARGE_CAP', 'PS_DISCHARGE_CAP', 'PS_CHAR_WATTHOUR', 'PS_AVG_CURRENT', 'PS_AVG_VOLTAGE']:
        if column in df.columns:
            df[column] = df[column] / 1000

    # Set PS_WATT to negative if PS_CURRENT is negative
    if 'PS_CURRENT' in df.columns and 'PS_WATT' in df.columns:
        df.loc[df['PS_CURRENT'] < 0, 'PS_WATT'] = df['PS_WATT'].abs() * -1

    # step end dataframe은 cts에서 가져올 것이라 일단 주석
    # Extract rows where PS_STEP_TIME resets to '00:00:00.0000'
    # if 'PS_STEP_TIME' in df.columns:
    #     # Identify rows where PS_STEP_TIME is less than the previous row, indicating a reset
    #     step_end_df = df[df['PS_STEP_TIME'].astype('timedelta64[s]').diff().shift(-1) < pd.Timedelta(0)]
    #     step_end_df = step_end_df.append(df.iloc[-1,:])
    
    # if 'PS_STEP_TIME' in step_end_df.columns:
    #     step_end_df['PS_STEP_TIME'] = pd.to_datetime(step_end_df['PS_STEP_TIME'], unit='s').dt.strftime('%H:%M:%S.%f').str[:-4]

    return df
    
def parse_binary_file(file_path):
    with open(file_path, 'rb') as file:
        # Read headers
        file_id_header = _read_ps_file_id_header(file)
        record_file_header = _read_ps_record_file_header(file)
        # Read record data
        print(file_id_header)
        print(record_file_header)
        nColumCount = record_file_header['nColumCount']
        record_data = _read_record_data(file, nColumCount)
        processed_data = _preprocess_data(record_data, record_file_header['columnNames'])
        
        return {
            'FileIDHeader': file_id_header,
            'RecordFileHeader': record_file_header,
            'ProcessedData' : processed_data
        }

In [20]:
file_path = r'C:\Users\jhchoei\Desktop\Workspace\TOS_Crawler\data\tos\07001792_20250715_07001792_JF2S_DV1_T25_05CP_UECSF10075\M01Ch020[020]\07001792_20250715_07001792_JF2S_DV1_T25_05CP_UECSF10075.cyc'  # 바이너리 파일의 경로를 입력하세요.


In [21]:
parsed_data = parse_binary_file(file_path)

{'szFileID': 20240905, 'szFileVersion': 4103, 'szCreateDateTime': '2025-07-15 13:48:15.777', 'szDescription': 'PNE record data file.', 'szReserved': ''}
{'nColumCount': 44, 'awColumnItem': {'PS_DATA_SEQ': 22, 'PS_STEP_TIME': 6, 'PS_CV_END_TIME': 38, 'PS_TOT_TIME_CARRY': 40, 'PS_VOLTAGE': 1, 'PS_CURRENT': 2, 'PS_CAPACITY': 3, 'PS_INTEGRAL_CAPACITY': 36, 'PS_CHARGE_CAP': 26, 'PS_DISCHARGE_CAP': 27, 'PS_WATT_HOUR': 11, 'PS_INTEGRAL_WATTHOUR': 37, 'PS_CHAR_WATTHOUR': 34, 'PS_DISCHAR_WATTHOUR': 35, 'PS_AVG_CURRENT': 23, 'PS_AVG_VOLTAGE': 24, 'PS_TEMPERATURE': 12, 'PS_IMPEDANCE': 4, 'PS_WATT': 10, 'PS_CAPACITY_SUM': 25, 'PS_WATTHOUR_SUM': 33, 'PS_FARAD': 41, 'PS_CHAMBER_TEMPERATURE': 50, 'PS_TEMPERATURE2': 42, 'PS_CHARGE_CC_CAP': 44, 'PS_CHARGE_CV_CAP': 45, 'PS_DISCHARGE_CC_CAP': 46, 'PS_DISCHARGE_CV_CAP': 47, 'PS_IMPEDANCE_100MS': 75, 'PS_IMPEDANCE_1S': 76, 'PS_IMPEDANCE_5S': 77, 'PS_IMPEDANCE_30S': 78, 'PS_IMPEDANCE_60S': 79, 'PS_AMBIENT_TEMP': 73, 'PS_GAS_VOLTAGE': 74, 'PS_REALDATE': 48, 

In [22]:
parsed_data["ProcessedData"]

,PS_DATA_SEQ,PS_STEP_TIME,PS_CV_END_TIME,PS_TOT_TIME_CARRY,PS_VOLTAGE,PS_CURRENT,PS_CAPACITY,PS_INTEGRAL_CAPACITY,PS_CHARGE_CAP,PS_DISCHARGE_CAP,...,PS_GAS_VOLTAGE,PS_REALDATE,PS_REALCLOCK,PS_STATE,PS_RESTORED_FLAG,PS_CHAMBER_TEMP_SV,PS_CHILER_TEMP_PV,PS_CHILER_TEMP_SV,PS_CHILER_PUMP_PV,PS_CHILER_PUMP_SV
0,0.0,1.0,0.00000,0.0,0.0,3.263290,0.0,0.000000,0.0,0.000000,...,0.0,0.0,250715.0,134815808.0,2.0,0.0,25.0,999.0,999.0,999.0
1,999.0,2.0,60.00000,0.0,0.0,3.263261,0.0,0.000000,0.0,0.000000,...,0.0,0.0,250715.0,134915840.0,2.0,0.0,25.0,999.0,999.0,999.0
2,999.0,3.0,120.00000,0.0,0.0,3.263290,0.0,0.000000,0.0,0.000000,...,0.0,0.0,250715.0,135015840.0,2.0,0.0,25.0,999.0,999.0,999.0
3,999.0,4.0,180.00000,0.0,0.0,3.263290,0.0,0.000000,0.0,0.000000,...,0.0,0.0,250715.0,135115840.0,2.0,0.0,25.0,999.0,999.0,999.0
4,999.0,5.0,240.00000,0.0,0.0,3.263189,0.0,0.000000,0.0,0.000000,...,0.0,0.0,250715.0,135215840.0,2.0,0.0,25.0,999.0,999.0,999.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602077,999.0,602078.0,1452.51001,0.0,0.0,3.263391,0.0,30827.898438,0.0,30.827898,...,0.0,0.0,250827.0,172241632.0,3.0,0.0,25.0,999.0,999.0,999.0
602078,999.0,602079.0,1452.51001,0.0,0.0,3.263391,0.0,30827.898438,0.0,30.827898,...,0.0,0.0,250827.0,172242624.0,3.0,0.0,25.0,999.0,999.0,999.0
602079,999.0,602080.0,1452.51001,0.0,0.0,3.263391,0.0,30827.898438,0.0,30.827898,...,0.0,0.0,250827.0,172243632.0,3.0,0.0,25.0,999.0,999.0,999.0
602080,999.0,602081.0,1452.51001,0.0,0.0,3.263290,0.0,30827.898438,0.0,30.827898,...,0.0,0.0,250827.0,172244624.0,3.0,0.0,25.0,999.0,999.0,999.0


In [5]:
col_dict = {
    'PS_AMBIENT_TEMP': 'AmbientTemp',
    'PS_AVG_CURRENT': 'AvgCurrent',
    'PS_AVG_VOLTAGE': 'AvgVoltage',
    'PS_CAPACITY': 'Capacity',
    'PS_CHAMBER_TEMPERATURE': 'ChamberTemperature',
    'PS_CHARGE_CAP': 'ChargeCapacity',
    'PS_CHARGE_CC_CAP': 'ChargeCCCapacity',
    'PS_CHAR_WATTHOUR': 'CharWatthour',
    'PS_CURRENT': 'Current',
    'PS_DATA_SEQ': 'DataSeq',  # 'No'는 'DataSeq'로 덮어쓰여짐
    'PS_DISCHARGE_CAP': 'DischargeCapacity',
    'PS_DISCHARGE_CC_CAP': 'DischargeCCCapacity',
    'PS_DISCHAR_WATTHOUR': 'DischarWatthour',
    'PS_GAS_VOLTAGE': 'GasVoltage',
    'PS_IMPEDANCE': 'Impedance',
    'PS_IMPEDANCE_100MS': 'Impedance100Ms',
    'PS_IMPEDANCE_1S': 'Impedance1S',
    'PS_IMPEDANCE_30S': 'Impedance30S',
    'PS_IMPEDANCE_5S': 'Impedance5S',
    'PS_IMPEDANCE_60S': 'Impedance60S',
    'PS_REALCLOCK': 'Realclock',
    'PS_REALDATE': 'Realdate',
    'PS_STEP_TIME': 'StepTime_sec',  # 'StepTime'은 'StepTime_sec'으로 덮어쓰여짐
    'PS_TEMPERATURE': 'Temp',
    'PS_VOLTAGE': 'Voltage',
    'PS_WATT': 'Power',  # 'Watt'은 'Power'로 덮어쓰여짐
    'PS_WATT_HOUR': 'WattHour'
}

In [ ]:
"CharWatthour", "ChargeCCCapacity"

In [51]:
temp = parsed_data["ProcessedData"].rename(columns=col_dict)
temp.filter(regex='PS')["PS_CV_END_TIME"].value_counts()

PS_CV_END_TIME
0.0    1589319
Name: count, dtype: int64

In [ ]:
parsed_data["ProcessedData"]["PS_DATA_SEQ"]

0                1.0
1                2.0
2                3.0
3                4.0
4                5.0
             ...    
1589314    1600160.0
1589315    1600161.0
1589316    1600162.0
1589317    1600163.0
1589318    1600164.0
Name: PS_DATA_SEQ, Length: 1589319, dtype: float64

In [ ]:
import sys
sys.path.append(r"C:\Users\jhchoei\Desktop\Workspace\TOS")
from processor.PNE import CycProcessor
from assets import config_prevdd

processor = CycProcessor(config_prevdd)
results = processor.parse_binary_file(file_path)


In [12]:
results["ProcessedData"].to_parquet("ds.parquet", engine='pyarrow', index=False, compression='snappy')

In [10]:
results["ProcessedData"]

,PS_DATA_SEQ,PS_STEP_TIME,PS_VOLTAGE,PS_CURRENT,PS_CAPACITY,PS_CHARGE_CAP,PS_DISCHARGE_CAP,PS_WATT_HOUR,PS_CHAR_WATTHOUR,PS_DISCHAR_WATTHOUR,...,PS_CHAMBER_TEMPERATURE,PS_CHARGE_CC_CAP,PS_DISCHARGE_CC_CAP,PS_IMPEDANCE_100MS,PS_IMPEDANCE_1S,PS_IMPEDANCE_5S,PS_IMPEDANCE_30S,PS_IMPEDANCE_60S,PS_AMBIENT_TEMP,PS_GAS_VOLTAGE
0,1.0,0.000000,3.331181,0.0,0.000000,0.000000,0.0,0.000,0.000000,0.0,...,45.0,0.000000,0.0,0.000,0.000,0.000,0.000,0.000,42.333000,2.0
1,2.0,60.000000,3.331181,0.0,0.000000,0.000000,0.0,0.000,0.000000,0.0,...,45.0,0.000000,0.0,0.000,0.000,0.000,0.000,0.000,42.433998,2.0
2,3.0,120.000000,3.331181,0.0,0.000000,0.000000,0.0,0.000,0.000000,0.0,...,45.0,0.000000,0.0,0.000,0.000,0.000,0.000,0.000,42.433998,2.0
3,4.0,180.000000,3.331181,0.0,0.000000,0.000000,0.0,0.000,0.000000,0.0,...,45.0,0.000000,0.0,0.000,0.000,0.000,0.000,0.000,42.535000,2.0
4,5.0,240.000000,3.331181,0.0,0.000000,0.000000,0.0,0.000,0.000000,0.0,...,45.0,0.000000,0.0,0.000,0.000,0.000,0.000,0.000,42.333000,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1589314,1600160.0,2133.800049,3.297516,0.0,45.730137,45730.136719,0.0,150.957,150.957001,0.0,...,45.0,45730.136719,0.0,0.661,0.738,0.912,1.662,2.314,42.333000,2.0
1589315,1600161.0,2133.800049,3.297516,0.0,45.730137,45730.136719,0.0,150.957,150.957001,0.0,...,45.0,45730.136719,0.0,0.661,0.738,0.912,1.662,2.314,42.333000,2.0
1589316,1600162.0,2133.800049,3.297516,0.0,45.730137,45730.136719,0.0,150.957,150.957001,0.0,...,45.0,45730.136719,0.0,0.661,0.738,0.912,1.662,2.314,42.333000,2.0
1589317,1600163.0,2133.800049,3.297516,0.0,45.730137,45730.136719,0.0,150.957,150.957001,0.0,...,45.0,45730.136719,0.0,0.661,0.738,0.912,1.662,2.314,42.333000,2.0
